In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import numpy as np

NB_ID = "15"


In [ ]:
DATA_PATH = get_unet_path(STAGE_CLEANED, signal=BarrageSignal)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Could not find {DATA_PATH}. Did you run Notebook 14?")

print(f"Loading data from {DATA_PATH}...")
barrage_tensor = np.load(DATA_PATH)
print(f"Loaded Shape: {barrage_tensor.shape}")

In [ ]:
# --- Physics Verification ---
# Pick a random file index to test
idx = np.random.randint(0, len(barrage_tensor))
sample_i = barrage_tensor[idx, 0, :]
sample_q = barrage_tensor[idx, 1, :]
sample_complex = sample_i + 1j * sample_q

# Power Calculation
# Power = Mean of Magnitude Squared
power = np.mean(np.abs(sample_complex)**2)

# DC Offset Calculation
# Offset = Magnitude of the Mean
dc_offset = np.abs(np.mean(sample_complex))

print(f"\n--- QC Report (Sample #{idx}) ---")
print(f"Target Power: 1.0  | Actual: {power:.5f}")
print(f"Target DC:    0.0  | Actual: {dc_offset:.5f}")

if 0.95 < power < 1.05:
    print("Power Physics: PASS")
else:
    print("Power Physics: FAIL")

# --- Visualization ---
plt.figure(figsize=(18, 6))

# Plot 1: Time Domain
# Standard: "In-Phase & Quadrature Components"
plt.subplot(1, 3, 1)
plt.plot(sample_i[:500], label='In-Phase (I)', alpha=0.8, linewidth=1)
plt.plot(sample_q[:500], label='Quadrature (Q)', alpha=0.8, linewidth=1)
plt.title(f"Time Domain Analysis")
plt.xlabel("Sample Index (n)")
plt.ylabel("Amplitude")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)

# Plot 2: Constellation Diagram
# Standard: "IQ Plane" or "Signal Constellation"
plt.subplot(1, 3, 2)
# Use a heatmap-like scatter or just smaller points for clarity
plt.scatter(sample_i[:5000], sample_q[:5000], s=1, alpha=0.2, color='#6a0dad', label='Noise Samples')
plt.title("Constellation Diagram (Complex Plane)")
plt.xlabel("In-Phase Amplitude (I)")
plt.ylabel("Quadrature Amplitude (Q)")
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", markerscale=5)

# Add Reference Unit Circle (RMS Power = 1.0)
circle = plt.Circle((0, 0), 1, color='red', fill=False, linestyle='--', linewidth=1.5, label='RMS Unit Circle')
plt.gca().add_patch(circle)

# Plot 3: Power Spectral Density (PSD)
# Standard: "PSD Estimate (Welch's Method)"
plt.subplot(1, 3, 3)
plt.psd(sample_complex, NFFT=1024, Fs=25e6, color='green', linewidth=1)
plt.title("Power Spectral Density (PSD)")
plt.xlabel("Frequency (Hz)") 
plt.ylabel("Power Spectral Density (dB/Hz)")
plt.grid(True, alpha=0.3)
plt.tight_layout()

save_plot(f"barrage_physics_check_sample_{idx}", machine_id="B", nb_id=NB_ID, fig_id="01")
plt.show()